In [ ]:
import os
import glob
import time
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from speechbrain.inference.speaker import EncoderClassifier

# ==============================================================================
# 1. THIẾT LẬP THIẾT BỊ VÀ TỰ ĐỘNG TÌM ĐƯỜNG DẪN DATASET
# ==============================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"• GPU: {torch.cuda.get_device_name(0)}")

KAGGLE_INPUT = '/kaggle/input'
dataset_dir = glob.glob(os.path.join(KAGGLE_INPUT, '**/dataset'), recursive=True)
DATASET_PATH = dataset_dir[0] if dataset_dir else os.path.join(KAGGLE_INPUT, 'speaker-verification-adaptive-augmentat/dataset')

print(f"• Dataset Path: {DATASET_PATH}")

# ==============================================================================
# 2. QUÉT FILE & GÁN NHÃN SIÊU TỐC (BỎ QUA TÍNH DUNG LƯỢNG)
# ==============================================================================
audio_files = glob.glob(os.path.join(DATASET_PATH, '**/*.wav'), recursive=True)
print(f"• Tìm thấy tổng cộng: {len(audio_files)} file .wav")

# Gom thông tin đường dẫn và ID người nói trực tiếp từ cấu trúc thư mục
data_info = [{'path': p, 'speaker_id': os.path.basename(os.path.dirname(p))} for p in audio_files]
df = pd.DataFrame(data_info)

# Lọc các speaker có từ 2 file audio trở lên để chia Train/Val hợp lệ
spk_counts = df['speaker_id'].value_counts()
valid_speakers = spk_counts[spk_counts >= 2].index

filtered_df = df[df['speaker_id'].isin(valid_speakers)].copy()

unique_speakers = sorted(filtered_df['speaker_id'].unique())
label2id = {spk: idx for idx, spk in enumerate(unique_speakers)}
id2label = {idx: spk for spk, idx in label2id.items()}
filtered_df['label'] = filtered_df['speaker_id'].map(label2id)

NUM_CLASSES = len(unique_speakers)
print(f"• Số lớp huấn luyện (Speakers): {NUM_CLASSES}")

train_df, val_df = train_test_split(
    filtered_df, test_size=0.2, random_state=42, stratify=filtered_df['label']
)

print(f"• Tập Train: {len(train_df)} mẫu | Tập Val: {len(val_df)} mẫu")

# ==============================================================================
# 3. DATASET & DATALOADER
# ==============================================================================
class FastKaggleDataset(Dataset):
    def __init__(self, df, target_sr=16000, segment_len=3.0, is_train=True):
        self.df = df.reset_index(drop=True)
        self.target_sr = target_sr
        self.segment_samples = int(segment_len * target_sr)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def _load_audio(self, path):
        wav, sr = torchaudio.load(path)
        if sr != self.target_sr:
            wav = T.Resample(sr, self.target_sr)(wav)
        if wav.shape[0] > 1:
            wav = torch.mean(wav, dim=0, keepdim=True)
        return wav.squeeze(0)

    def _crop_or_pad(self, wav):
        if len(wav) >= self.segment_samples:
            if self.is_train:
                start = random.randint(0, len(wav) - self.segment_samples)
            else:
                start = (len(wav) - self.segment_samples) // 2
            wav = wav[start:start + self.segment_samples]
        else:
            padding = self.segment_samples - len(wav)
            wav = F.pad(wav, (0, padding))
        return wav

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav = self._load_audio(row['path'])
        wav = self._crop_or_pad(wav)
        return wav, torch.tensor(row['label'], dtype=torch.long)

train_dataset = FastKaggleDataset(train_df, is_train=True)
val_dataset = FastKaggleDataset(val_df, is_train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

# ==============================================================================
# 4. MÔ HÌNH ECAPA-TDNN
# ==============================================================================
class ECAPAFinetuneModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        device_str = "cuda:0" if torch.cuda.is_available() else "cpu"
        self.spk_classifier = EncoderClassifier.from_hparams(
            source="speechbrain/spkrec-ecapa-voxceleb",
            run_opts={"device": device_str}
        )
        self.backbone = self.spk_classifier.mods.embedding_model
        self.fc = nn.Linear(192, num_classes)

    def forward(self, wavs):
        feats = self.spk_classifier.mods.compute_features(wavs)
        lengths = torch.ones(feats.shape[0], device=feats.device)
        feats = self.spk_classifier.mods.mean_var_norm(feats, lengths)

        embeddings = self.backbone(feats, lengths=lengths).squeeze(1)
        logits = self.fc(embeddings)

        return logits, embeddings

model = ECAPAFinetuneModel(num_classes=NUM_CLASSES).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)



• GPU: Tesla T4
• Dataset Path: /kaggle/input/datasets/lequanglinh/speaker-verification-adaptive-augmentation-dataset/dataset
• Tìm thấy tổng cộng: 125847 file .wav
• Số lớp huấn luyện (Speakers): 1547
• Tập Train: 100575 mẫu | Tập Val: 25144 mẫu


hyperparams.yaml: 0.00B [00:00, ?B/s]

embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

mean_var_norm_emb.ckpt:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

classifier.ckpt:   0%|          | 0.00/5.53M [00:00<?, ?B/s]

label_encoder.txt: 0.00B [00:00, ?B/s]

In [ ]:
EPOCHS = 20
SAVE_PATH = '/kaggle/working/ecapa_tdnn_finetuned_3.pt'
best_val_acc = 0.0

print("\n🚀 Bắt đầu huấn luyện...")

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()

    # Train Phase
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for wavs, labels in train_loader:
        wavs, labels = wavs.to(device), labels.to(device)

        optimizer.zero_grad()
        logits, _ = model(wavs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * wavs.size(0)
        preds = torch.argmax(logits, dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_epoch_loss = train_loss / train_total
    train_epoch_acc = train_correct / train_total

    # Val Phase
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for wavs, labels in val_loader:
            wavs, labels = wavs.to(device), labels.to(device)
            logits, _ = model(wavs)
            loss = criterion(logits, labels)

            val_loss += loss.item() * wavs.size(0)
            preds = torch.argmax(logits, dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_epoch_loss = val_loss / val_total
    val_epoch_acc = val_correct / val_total

    scheduler.step(val_epoch_loss)
    elapsed = time.time() - start_time

    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({elapsed:.1f}s) | "
          f"Train Loss: {train_epoch_loss:.4f} - Acc: {train_epoch_acc*100:.2f}% | "
          f"Val Loss: {val_epoch_loss:.4f} - Acc: {val_epoch_acc*100:.2f}%")

    if val_epoch_acc > best_val_acc:
        best_val_acc = val_epoch_acc
        try:
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'label2id': label2id,
                'id2label': id2label,
                'val_acc': best_val_acc
            }
            torch.save(checkpoint, SAVE_PATH)
            print(f"-> Saved best checkpoint at {SAVE_PATH} (Acc: {best_val_acc*100:.2f}%)")
        except Exception as e:
            print(f"-> Error saving checkpoint: {e}")


🚀 Bắt đầu huấn luyện...
Epoch [01/20] (829.1s) | Train Loss: 18.6985 - Acc: 21.59% | Val Loss: 8.2999 - Acc: 46.63%
-> Saved best checkpoint at /kaggle/working/ecapa_tdnn_finetuned_3.pt (Acc: 46.63%)
Epoch [02/20] (704.0s) | Train Loss: 5.3598 - Acc: 59.01% | Val Loss: 3.9727 - Acc: 67.21%
-> Saved best checkpoint at /kaggle/working/ecapa_tdnn_finetuned_3.pt (Acc: 67.21%)
Epoch [03/20] (704.5s) | Train Loss: 2.6660 - Acc: 74.03% | Val Loss: 2.6148 - Acc: 75.42%
-> Saved best checkpoint at /kaggle/working/ecapa_tdnn_finetuned_3.pt (Acc: 75.42%)
Epoch [04/20] (701.9s) | Train Loss: 1.5679 - Acc: 81.70% | Val Loss: 1.9973 - Acc: 79.42%
-> Saved best checkpoint at /kaggle/working/ecapa_tdnn_finetuned_3.pt (Acc: 79.42%)
Epoch [05/20] (687.1s) | Train Loss: 0.9863 - Acc: 86.38% | Val Loss: 1.6423 - Acc: 82.00%
-> Saved best checkpoint at /kaggle/working/ecapa_tdnn_finetuned_3.pt (Acc: 82.00%)
Epoch [06/20] (697.0s) | Train Loss: 0.6520 - Acc: 89.66% | Val Loss: 1.4791 - Acc: 83.29%
-> Saved